# Transformer Architecture

把注意力装进完整架构：位置编码、残差 + LayerNorm + FFN 的 Encoder 块、Decoder 的因果掩码。本课实现一个可用的 Transformer Block。


## 0. 环境配置与导入


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib

matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Arial Unicode MS", "Microsoft YaHei", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)


## 1. 为什么需要位置编码


注意力是**置换等变**的：打乱 token 顺序，输出只是跟着打乱——模型完全不知道"谁在前谁在后"。必须显式注入位置信息。

**正弦位置编码**（原始 Transformer）：

$$PE_{(pos, 2i)} = \sin\Big(\frac{pos}{10000^{2i/d}}\Big), \qquad PE_{(pos, 2i+1)} = \cos\Big(\frac{pos}{10000^{2i/d}}\Big)$$

频率随维度递减：低维编码位置、高维编码精细偏移。


In [ ]:
def sinusoidal_pe(T, d_model):
    pe = torch.zeros(T, d_model)
    pos = torch.arange(T).unsqueeze(1).float()
    i = torch.arange(d_model // 2).float()
    ang = pos / 10000 ** (2*i / d_model)
    pe[:, 0::2] = torch.sin(ang)
    pe[:, 1::2] = torch.cos(ang)
    return pe

pe = sinusoidal_pe(64, 64)
plt.figure(figsize=(9, 4))
plt.imshow(pe.numpy().T, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
plt.xlabel('位置 pos'); plt.ylabel('维度 i'); plt.colorbar(label='编码值')
plt.title('正弦位置编码热力图（低维慢变、高维快变）')


In [ ]:
# 关键性质：相对位置点积只与偏移 k 有关
pe = sinusoidal_pe(128, 128)
for k in [1, 3, 7]:
    vals = np.array([(pe[p+k] @ pe[p]).item() for p in range(0, 128-k, 4)])
    print(f"k={k}: PE(pos+k)·PE(pos) 的极差 = {np.ptp(vals):.2e}（≈0，只依赖 k）")
print("→ 正弦编码让模型天然感知『相对位置』")


## 2. Transformer Block（Pre-LN）


Encoder 块 = 多头自注意力 + 前馈网络，每个都套"残差 + LayerNorm"：

$$x \gets x + \text{Dropout}(\text{Attn}(\text{LN}(x))) \\
x \gets x + \text{Dropout}(\text{FFN}(\text{LN}(x)))$$

**Pre-LN**（先归一化再子层）：比原始 Post-LN 更稳，梯度更干净。FFN 是"逐位置的两层 MLP"（注意力的输出再经非线性变换）。


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model, self.n_heads = d_model, n_heads
        self.d_k = d_model // n_heads
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)
    def forward(self, x, mask=None):
        B, T, _ = x.shape
        Q = self.Wq(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.Wk(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.Wv(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        out = attn @ V
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.Wo(out)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        x = x + self.dropout(self.attn(self.norm1(x), mask))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x

x = torch.randn(2, 10, 32)
block = TransformerBlock(32, 4, 64)
print("Block 输出形状:", tuple(block(x).shape), "（B, T, d_model 不变 ✓）")


## 3. 因果掩码：Decoder 看不到未来


自回归生成时，位置 $t$ 只能看 $< t$ 的 token。实现：把上三角的注意力分数设为 $-\infty$，softmax 后权重为 0。


In [ ]:
T = 6
causal = torch.tril(torch.ones(T, T, dtype=torch.bool))
scores = torch.randn(1, 1, T, T)
masked = scores.masked_fill(~causal, float('-inf'))
probs = torch.softmax(masked, dim=-1)

plt.figure(figsize=(5, 4.2))
plt.imshow(probs[0, 0].numpy(), cmap='Blues')
plt.xticks(range(T)); plt.yticks(range(T))
plt.xlabel('key 位置'); plt.ylabel('query 位置')
plt.title('因果掩码后的注意力：上三角全 0')
plt.colorbar()
print("第 4 行（query=4）只关注位置 ≤ 4:", np.round(probs[0,0,4].numpy(), 3))


## 4. 完整 Encoder 栈


In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, n_layers, dropout=0.1):
        super().__init__()
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x, mask=None):
        for blk in self.blocks:
            x = blk(x, mask)
        return self.norm(x)

enc = TransformerEncoder(32, 4, 64, 3)
print("3 层 Encoder 输出:", tuple(enc(x).shape))
print(f"总参数: {sum(p.numel() for p in enc.parameters()):,}")


## 5. 为什么 LayerNorm 而不是 BatchNorm


- **BN 统计整批**：序列长度/样本变化时统计不稳定，且 NLP 里 batch 小
- **LN 统计每个样本自身**：对每个 token 的 $d_{model}$ 维做归一化（$\mu, \sigma$ 是逐样本逐位置算的）——与 batch 无关，天然适配变长序列

BN 归一化"跨样本"，LN 归一化"跨特征"——一个管批量、一个管尺度。


In [ ]:
x = torch.randn(4, 6, 32)
ln = nn.LayerNorm(32)
y = ln(x)
# 验证：每个 (样本, 位置) 的 32 维均值为 0、方差为 1
print(f"逐 token 均值: {y.mean(-1).abs().max():.2e}，方差: {y.var(-1).max():.3f}")
print("→ LayerNorm 对每个 token 独立归一化，与 batch 无关")


## 课后练习


1. **验证**：证明 $PE_{(pos,2i)}$ 是周期函数，周期随 $i$ 增大而增大。
2. **Post-LN vs Pre-LN**：把 Block 改成先子层后归一化（Post-LN），训练对比稳定性。
3. **位置编码对比**：试可学习位置编码（nn.Parameter），比较训练曲线。
4. **掩码注意力**：写一个只允许"每 2 个位置"关注的稀疏掩码，观察信息流。
5. **思考**：为什么 FFN 用"升维再降维"（d_ff 通常 4×d_model）的结构？
